In [1]:
import os
from typing import TypedDict, Annotated, Optional, Literal
from operator import add
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini")

In [2]:
class AdaptiveState(TypedDict):
    query: str
    route: str
    documents: Annotated[list[str], add]
    web_results: Annotated[list[str], add]
    sub_queries: Annotated[list[str], add]
    retrieved: Annotated[list[str], add]
    hop_count: int
    max_hops: int
    answer: str

def init_state(q: str, max_hops: int = 3) -> dict:
    return {"query": q, "route": "", "documents": [], "web_results": [],
            "sub_queries": [], "retrieved": [], "hop_count": 0,
            "max_hops": max_hops, "answer": ""}


In [3]:
class _Route(BaseModel):
    """쿼리를 처리 경로로 분류."""
    route: Literal["direct", "rag", "web", "multihop"] = Field(
        description="direct=상식, rag=사내문서, web=최신정보, multihop=복합추론"
    )

router_llm = llm.with_structured_output(_Route)

def classify_node(state: AdaptiveState) -> dict:
    sys = ("질문을 분류하세요. direct=LLM 상식, rag=사내 문서, "
           "web=최신/실시간, multihop=여러 단계 추론이 필요한 복합 질문.")
    r = router_llm.invoke([SystemMessage(content=sys),
                            HumanMessage(content=state["query"])])
    return {"route": r.route}

In [4]:
builder = StateGraph(AdaptiveState)
builder.add_node("classify", classify_node)
builder.add_edge(START, "classify")
builder.add_edge("classify", END)
app_v2 = builder.compile()

queries = [
    "파이썬 리스트가 뭐야?",
    "우리 회사 휴가 정책은?",
    "오늘 코스피 지수는?",
    "봉준호 2019년 영화의 음악감독 출생연도는?",
]
for q in queries:
    r = app_v2.invoke(init_state(q))
    print(f"{q[:25]:<25} → route={r['route']}")

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=_Route(route='direct'), input_type=_Route])
  return self.__pydantic_serializer__.to_python(


파이썬 리스트가 뭐야?              → route=direct


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=_Route(route='rag'), input_type=_Route])
  return self.__pydantic_serializer__.to_python(


우리 회사 휴가 정책은?             → route=rag


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=_Route(route='web'), input_type=_Route])
  return self.__pydantic_serializer__.to_python(


오늘 코스피 지수는?               → route=web
봉준호 2019년 영화의 음악감독 출생연도는? → route=direct


In [5]:
def direct_node(state: AdaptiveState) -> dict:
    r = llm.invoke([
        SystemMessage(content="간결히 답변하세요."),
        HumanMessage(content=state["query"]),
    ])
    return {"answer": r.content}

def route_fn(state: AdaptiveState) -> str:
    return state["route"]

In [6]:
builder = StateGraph(AdaptiveState)
builder.add_node("classify", classify_node)
builder.add_node("direct", direct_node)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_fn, {
    "direct": "direct",
    "rag": END,       # 아직 미구현
    "web": END,
    "multihop": END,
})
builder.add_edge("direct", END)
app_v3 = builder.compile()

# direct로 분류될 쿼리만 진짜로 답함
r = app_v3.invoke(init_state("파이썬 리스트가 뭐야?"))
print(f"route={r['route']}\nanswer={r['answer'][:80]}")

route=direct
answer=파이썬 리스트는 여러 개의 값을 순서대로 저장할 수 있는 데이터 구조입니다. 리스트는 대괄호 `[]`로 정의하며, 각 요소는 쉼표로 구분됩니다.


In [7]:
DOCS = [
    "휴가 정책: 입사 1년 미만은 월 1일, 1년 이상은 연 15일 유급 휴가가 부여됩니다.",
    "출장 규정: 국내 출장은 일비 5만원, 해외는 일비 10만원이 지급됩니다.",
    "재택근무: 주 2일까지 재택근무 가능. 팀장 사전 승인 필요.",
    "야근 식대: 오후 8시 이후 야근 시 1만원 식대 지급.",
    "교육비 지원: 직무 관련 도서/강의 연 30만원 한도 지원.",
    "건강검진: 매년 1회 종합검진 무료 제공.",
    "복장 규정: 비즈니스 캐주얼. 고객 미팅 시 정장 권장.",
]


In [8]:
docs = [Document(page_content=d) for d in DOCS]
vstore = FAISS.from_documents(docs, OpenAIEmbeddings())
retriever = vstore.as_retriever(search_kwargs={'k' : 2})

In [9]:
len(docs)

7

In [ ]:
# rag_node가 검색된 문서의 개수를 print 로그로 남기게 수정을 해주세요

In [10]:
def rag_node(state: AdaptiveState) -> dict:
    hits = retriever.invoke(state['query'])
    ctx = '\n'.join(h.page_content for h in hits)
    r = llm.invoke([
        SystemMessage(content = f'다음 사내 문서 근거로 답변해주세요:\n{ctx}'),
        HumanMessage(content = state['query'])
    ])
    return {
        'document' : [h.page_content for h in hits], 
        'answer' : '[rag] ' + r.content
    }

In [11]:
builder = StateGraph(AdaptiveState)
builder.add_node("classify", classify_node)
builder.add_node("direct", direct_node)
builder.add_node('rag', rag_node)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_fn, {
    "direct": "direct",
    "rag": 'rag',       
    "web": END,
    "multihop": END,
})
builder.add_edge("direct", END)
builder.add_edge('rag', END)
app_v3 = builder.compile()

r = app_v3.invoke(init_state("우리 회사 휴가 정책은?"))

In [13]:
r

{'query': '우리 회사 휴가 정책은?',
 'route': 'rag',
 'documents': [],
 'web_results': [],
 'sub_queries': [],
 'retrieved': [],
 'hop_count': 0,
 'max_hops': 3,
 'answer': '[rag] 우리 회사의 휴가 정책은 다음과 같습니다: \n\n- 입사 1년 미만인 직원은 매월 1일의 유급 휴가가 부여됩니다.\n- 입사 1년 이상인 직원은 연간 15일의 유급 휴가가 부여됩니다.'}

In [12]:
r['route']

'rag'